# 05 · Ejercicios extra — fuentes infrautilizadas y wrangling

**Tiempo estimado:** 30-45 min (backup, si quedan ganas).

**Objetivos.** Practicar transformaciones de pandas sobre las fuentes de datos que los notebooks 01-03 apenas tocan:

- **`anuario_aforos`**: hay 135 estaciones de aforo, no solo Pinos-Genil.
- **AEMET**: el JSON crudo tiene 24 columnas (temperaturas, viento, radiación, presión, humedad y horas de cada pico); el resto de la sesión usa solo `prec`.
- **SAIH horario**: el Excel `HistSAIH.xlsx` tiene resolución horaria 2018-2026, incluida la crecida de feb-2026.

Técnicas pandas que aparecen: `pivot_table`, alineación con `pd.concat({...}, axis=1)`, broadcasting con `.div(axis=1)`, `pd.cut`, `pd.crosstab(normalize=...)`, `.map(dict)`, `rolling(...).sum().shift()` para detectar eventos, `groupby().agg(...)` multi-función con tuplas, `pd.to_datetime(format=...)` + `.dt.hour`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})

## A1 · Matriz de cobertura estación × año

`afliq.csv` reúne el caudal diario de **135 estaciones del Guadalquivir**. Antes de elegir una estación para un estudio conviene ver **qué años tienen datos** — algunas series acaban en los 70, otras empiezan en los 90, otras tienen huecos importantes.

Esto es un caso de libro para `pivot_table` con `aggfunc="count"`: contamos días con dato por (estación, año) y obtenemos directamente la matriz de cobertura.


In [ ]:
afliq = pd.read_csv(
    "data/raw/anuario_aforos/GUADALQUIVIR_afliq.csv",
    sep=";",
    encoding="latin-1",
)
afliq.columns = [c.strip().lower() for c in afliq.columns]
afliq["fecha"] = pd.to_datetime(afliq["fecha"], format="%d/%m/%Y", errors="coerce")
afliq["anio"] = afliq["fecha"].dt.year

cobertura = afliq.pivot_table(index="indroea", columns="anio", values="caudal", aggfunc="count")
print(f"Estaciones: {cobertura.shape[0]}   años: {cobertura.shape[1]}")
print(f"Periodo: {int(cobertura.columns.min())} → {int(cobertura.columns.max())}")

In [ ]:
# Heatmap de las 30 estaciones con más datos
top30 = cobertura.sum(axis=1).nlargest(30).index
cob30 = cobertura.loc[top30]

fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(
    cob30.values,
    aspect="auto",
    cmap="viridis",
    extent=[
        cob30.columns.min() - 0.5,
        cob30.columns.max() + 0.5,
        len(cob30) - 0.5,
        -0.5,
    ],
)
ax.set_yticks(range(len(cob30)))
ax.set_yticklabels(cob30.index, fontsize=8)
ax.set_xlabel("Año")
ax.set_ylabel("ROEA")
ax.set_title("Cobertura del Anuario (días con dato) — top 30 estaciones del Guadalquivir")
plt.colorbar(im, ax=ax, label="días/año")
plt.tight_layout()

**Lectura:**

- Bandas verticales oscuras → años "malos" sistémicos (cierre administrativo, sequías que tumbaron sensores…).
- Filas mayormente oscuras → estaciones que ya no operan. Filas claras al final → estaciones modernas / SAIH.
- El **periodo común** a la mayoría es típicamente 1980-2010. Si quieres un análisis multi-estación, este heatmap te ahorra perder tiempo eligiendo series incompatibles.

**Variantes:**

- Cambia `aggfunc="count"` por `aggfunc="mean"` para ver el **caudal medio** anual por estación (escala log recomendada).
- Reemplaza `nlargest(30)` por filtrar `cobertura.notna().sum(axis=1) >= 40` para quedarte solo con series de >40 años.


## A2 · Caudal específico (l/s/km²) por subcuenca

Comparar caudales brutos entre estaciones es engañoso: una estación que drena 100 km² no es comparable con una que drena 50.000 km². El **caudal específico** normaliza por área:

$$
q = \frac{Q \, [\text{m}^3/\text{s}] \cdot 1000}{A \, [\text{km}^2]} \; [\text{l/s/km}^2]
$$

Esto requiere combinar `afliq.csv` (caudal) con `estaf.csv` (catálogo de estaciones, columna `suprest` = superficie en km²). El patrón pandas: **`pivot` para alinear las series, broadcasting para dividir por una `Series` de áreas**.


In [ ]:
estaf = ud.cargar_anuario_estaciones().set_index("indroea")
print("Catálogo de estaciones —", estaf.shape, "columnas relevantes:")
print(estaf[["lugar", "suprest", "alti"]].head())

# Top 10 estaciones con más datos Y área conocida
top = afliq.groupby("indroea")["caudal"].count().sort_values(ascending=False)
top = [t for t in top.index if t in estaf.index and pd.notna(estaf.loc[t, "suprest"])][:10]
print(f"\nSeleccionadas: {top}")

In [ ]:
# Pivot wide: filas = fecha, columnas = ROEA, valores = caudal
wide = afliq.pivot_table(index="fecha", columns="indroea", values="caudal")
wide = wide[top]

# Caudal específico: dividir cada columna por su área correspondiente
areas = estaf.loc[top, "suprest"]
q_esp = wide.mul(1000).div(areas, axis=1)  # m³/s → l/s; dividido por km²

# Resumen por estación
resumen = pd.DataFrame(
    {
        "area_km2": areas,
        "Q_medio_m3s": wide.mean(),
        "q_esp_medio_lskm2": q_esp.mean(),
    }
).join(estaf["lugar"])
print(resumen.sort_values("q_esp_medio_lskm2", ascending=False).round(2))

In [ ]:
# Plot: caudal específico anual hidrológico (oct-sep) por estación
q_esp_anual = q_esp.resample("YE-SEP").mean()

fig, ax = plt.subplots(figsize=(11, 4))
q_esp_anual.plot(ax=ax, lw=1.1)
ax.set_ylabel("Caudal específico (l/s/km²)")
ax.set_xlabel("Año hidrológico")
ax.set_title("Caudal específico anual — top 10 ROEA del Guadalquivir")
ax.legend(title="ROEA", ncol=2, fontsize=8, loc="upper right")
plt.tight_layout()

**Lectura:**

- Las cabeceras de montaña (Sierras de Cazorla, Sierra Nevada) tienen **caudales específicos altos** (>20 l/s/km²) por la lluvia orográfica y el deshielo.
- Las estaciones de llanura aguas abajo de embalses tienen valores mucho menores: la regulación se traga la respuesta natural.
- Si dos estaciones tienen $q$ similar, hidrológicamente son "parientes" — aunque drenen áreas muy distintas.

**Técnica clave:** `wide.mul(1000).div(areas, axis=1)`. Pandas alinea columnas con el índice de `areas` automáticamente; broadcasting hace el resto. Sin pivot a wide, habría que hacer un merge fila a fila — orden de magnitud más lento.


## B3 · Días tipo: `pd.cut` + `crosstab`

`cargar_meteo_aemet` devuelve el DataFrame completo de Granada Aeropuerto 1995-2020 con 24 columnas. Aquí solo usamos `prec`, pero ya tienes a mano `tmin`, `tmax`, `sol`, `velmedia`, `racha`, `presMax/Min`, `hrMedia/Max/Min`…

Para clasificar cada día por intensidad de lluvia: `pd.cut` con cortes a medida.
Para resumir por estación del año: `pd.crosstab(..., normalize="index")`. La normalización por fila convierte cuentas en **proporciones por estación**, que es lo que tiene sentido comparar.


In [ ]:
meteo = ud.cargar_meteo_aemet()
print("Meteo AEMET Granada Aeropuerto:")
print(f"  Periodo : {meteo.index.min().date()} → {meteo.index.max().date()}")
print(f"  Días    : {len(meteo)}")
print(f"  Columnas: {list(meteo.columns)}")

In [ ]:
# Clasificación de días por intensidad de lluvia (mm/día)
bins = [-0.01, 0, 1, 5, 20, 1000]
labels = ["seco", "<1mm", "1-5mm", "5-20mm", ">20mm"]
meteo["cat_lluvia"] = pd.cut(meteo["prec"], bins=bins, labels=labels)

# Mes → estación del año (mapeo con .map(dict))
mes_a_estacion = {
    12: "DEF",
    1: "DEF",
    2: "DEF",
    3: "MAM",
    4: "MAM",
    5: "MAM",
    6: "JJA",
    7: "JJA",
    8: "JJA",
    9: "SON",
    10: "SON",
    11: "SON",
}
meteo["estacion_anyo"] = meteo.index.month.map(mes_a_estacion)

# Crosstab normalizada por fila → % de días de cada tipo por estación
tabla = pd.crosstab(meteo["estacion_anyo"], meteo["cat_lluvia"], normalize="index") * 100
tabla = tabla.reindex(["DEF", "MAM", "JJA", "SON"])
print("% de días por categoría de lluvia y estación del año:")
print(tabla.round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
tabla.plot.bar(stacked=True, ax=ax, colormap="YlGnBu", edgecolor="white")
ax.set_ylabel("% de días")
ax.set_xlabel("Estación del año")
ax.set_title("Días por categoría de lluvia — Granada Aeropuerto (1995-2020)")
ax.legend(title="lluvia/día", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

**Lectura:**

- En verano (JJA) más del 90% de los días son **secos**: clima mediterráneo de manual.
- Los días "torrenciales" (>20 mm) se concentran en otoño-invierno y son raros (<1% anual).
- La crosstab normalizada es **la** forma compacta de reportar "distribución condicional" en pandas — sustituye 10 líneas de groupby + reshape.

**Extensiones:**

- Cambiar `prec` por `tmax` con bins `[-50, 0, 15, 25, 35, 50]` → clasificar días por confort térmico.
- `pd.crosstab(meteo["cat_lluvia"], meteo["cat_tmax"], normalize="all")` → tabla 2D de cómo se combinan ambos extremos.


## B4 · Horarios de eventos extremos (polar plot)

Las columnas `horaXxx` de AEMET indican a qué hora ocurrió el pico de cada variable ese día. Vienen como strings `"HH:MM"`, a veces `"Varias"` o `None`. Hay que parsearlas defensivamente con `pd.to_datetime(..., format="%H:%M", errors="coerce")` y luego `.dt.hour`.

El resultado se visualiza en **proyección polar**: cada barra es un sector de 1 hora, su altura es la frecuencia.


In [ ]:
horas_tmax = pd.to_datetime(meteo["horatmax"], format="%H:%M", errors="coerce").dt.hour
horas_racha = pd.to_datetime(meteo["horaracha"], format="%H:%M", errors="coerce").dt.hour
horas_tmin = pd.to_datetime(meteo["horatmin"], format="%H:%M", errors="coerce").dt.hour

print("Días parseables por columna:")
print(f"  horatmax  : {horas_tmax.notna().sum()} / {len(meteo)}")
print(f"  horaracha : {horas_racha.notna().sum()} / {len(meteo)}")
print(f"  horatmin  : {horas_tmin.notna().sum()} / {len(meteo)}")
print("\nValores no-HH:MM en horaracha (muestra):")
no_parseable = meteo.loc[horas_racha.isna() & meteo["horaracha"].notna(), "horaracha"]
print(no_parseable.value_counts().head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), subplot_kw={"projection": "polar"})
datos = [
    (horas_tmax, "Pico de T_max", "#c2410c"),
    (horas_tmin, "Pico de T_min", "#1f6f8b"),
    (horas_racha, "Pico de racha viento", "#7c3aed"),
]
for ax, (h, titulo, color) in zip(axes, datos):
    counts = h.value_counts().reindex(range(24), fill_value=0)
    theta = np.deg2rad(counts.index * 15)  # 360°/24h = 15° por hora
    ax.bar(theta, counts.values, width=np.deg2rad(15), color=color, alpha=0.75, edgecolor="white")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 15)))
    ax.set_xticklabels([f"{h:02d}" for h in range(24)], fontsize=7)
    ax.set_title(titulo, pad=15)
plt.tight_layout()

**Lectura:**

- El pico de **T_max** está muy concentrado en torno a las 15-16h → el sol manda y la respuesta es casi determinista.
- El pico de **T_min** se concentra en las horas previas al amanecer (5-7h) → el enfriamiento radiativo nocturno también es predecible.
- El pico de **racha de viento**, en cambio, está **mucho más disperso**: hay tantos episodios sinópticos (frentes, tormentas) como locales (brisa, gota fría)… → mayor entropía, menos predecible.

**Técnica clave:**

- `pd.to_datetime(..., format="%H:%M", errors="coerce")` → strings mal formados (`"Varias"`, `None`) se convierten en `NaT` sin romper el pipeline.
- `.dt.hour` extrae el componente. Otros accesorios útiles: `.dt.dayofyear`, `.dt.dayofweek`, `.dt.quarter`.
- `.value_counts().reindex(range(24), fill_value=0)` rellena las horas sin observaciones a 0 — sin esto, el polar plot tendría "huecos".


## C1 · Eventos de lluvia discretos (SAIH horario)

El SAIH da lluvia y caudal **horarios** 2018-2026. Un análisis natural: descomponer la serie horaria de lluvia en **eventos** (rachas de horas con lluvia separadas por al menos 6 horas secas) y caracterizar cada uno por su duración, total acumulado, pico horario e intensidad media.

Combina dos patrones pandas potentes:

1. **Detección de eventos con `cumsum`**: una nueva ID empieza cuando se cumple cierta condición. `.where(filtro)` lo limita a las observaciones que pertenecen al evento.
2. **`groupby().agg(...)` con tuplas `(columna, función)`**: cuando se necesitan agregaciones distintas sobre columnas distintas en una sola pasada.


In [ ]:
saih = ud._cargar_saih_excel_horario()
P = saih["A20_202"].dropna()
print(f"Serie horaria de lluvia A20_202: {len(P):,} horas")
print(f"  Periodo: {P.index.min()} → {P.index.max()}")
print(f"  Horas con lluvia >0: {(P > 0).sum():,} ({(P > 0).mean():.1%})")

In [ ]:
# Un nuevo evento empieza cuando llueve TRAS al menos 6h secas consecutivas.
con_lluvia = P > 0
horas_secas_prev = (~con_lluvia).astype(int).rolling(6).sum().shift(1).fillna(6)
nuevo_evento = con_lluvia & (horas_secas_prev >= 6)

# cumsum sobre el flag → ID incremental de evento; .where(con_lluvia) → solo horas con lluvia
eventos_id = nuevo_evento.cumsum().where(con_lluvia)

# Resumen por evento con groupby().agg(...) usando tuplas (columna, función)
df_eventos = pd.DataFrame({"P": P, "ts": P.index})
resumen = df_eventos.groupby(eventos_id).agg(
    inicio=("ts", "min"),
    fin=("ts", "max"),
    horas_con_lluvia=("P", "count"),
    total_mm=("P", "sum"),
    pico_h_mm_h=("P", "max"),
)
resumen["duracion_h"] = ((resumen["fin"] - resumen["inicio"]) // pd.Timedelta("1h") + 1).astype(int)
resumen["intensidad_mm_h"] = resumen["total_mm"] / resumen["duracion_h"]

print(f"Total de eventos detectados: {len(resumen)}")
print("\nTop 10 eventos por total acumulado:")
print(
    resumen.nlargest(10, "total_mm")[
        ["inicio", "fin", "duracion_h", "total_mm", "pico_h_mm_h", "intensidad_mm_h"]
    ].round(1)
)

In [ ]:
# Distribución de tamaños de evento
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].hist(resumen["duracion_h"], bins=40, color="#1f6f8b", alpha=0.8)
axes[0].set_xlabel("Duración (h)")
axes[0].set_ylabel("Nº de eventos")
axes[0].set_yscale("log")
axes[0].set_title("Duración")

axes[1].hist(resumen["total_mm"], bins=40, color="#7c3aed", alpha=0.8)
axes[1].set_xlabel("Total acumulado (mm)")
axes[1].set_yscale("log")
axes[1].set_title("Acumulado por evento")

axes[2].scatter(resumen["duracion_h"], resumen["total_mm"], s=10, alpha=0.5, color="#c2410c")
axes[2].set_xlabel("Duración (h)")
axes[2].set_ylabel("Total (mm)")
axes[2].set_title("Duración vs Acumulado")
plt.tight_layout()

**Lectura:**

- La distribución de tamaños de evento es muy sesgada: la mayoría son chubascos cortos (<6h, <5mm); unos pocos episodios largos dominan el total anual.
- En el scatter duración-vs-acumulado se ve la doble naturaleza: eventos **cortos e intensos** (tormentas convectivas) vs **largos y suaves** (frentes).
- El pico de feb-2026 debe estar entre los top — comprueba con `resumen.loc[resumen["inicio"].dt.year == 2026]`.

**Técnicas clave:**

| Patrón | Idea |
|---|---|
| `rolling(6).sum().shift(1)` | "¿Cuántas horas secas hubo en las 6h anteriores?" |
| `cumsum()` sobre flag binario | Asigna ID incremental cada vez que el flag se activa |
| `.where(filtro)` | Mantiene los IDs solo donde aplica; `groupby` ignora los NaN |
| `agg(col=("c", "f"), ...)` | Múltiples agregaciones nombradas en una pasada |

Este combo (`rolling + cumsum + where + groupby.agg`) reaparece en mil sitios: detección de sesiones de usuario, racks de fallos consecutivos, episodios de contaminación atmosférica, periodos de funcionamiento de un equipo… vale la pena memorizarlo.

**Extensión:** repite el ejercicio sobre el caudal (`A20_211_X`) definiendo evento como "horas con $Q > P_{90}$". Resultado: catálogo de **avenidas** en lugar de tormentas.
